# 06 — Video Trimodal Fusion (Video + Text + Audio)
Fuses three modalities for action recognition:

| Modality | Source | Feature dim |
|---|---|---|
| **Visual** | CNN (ResNet50) + LSTM over 16 frames | 256 (projected) |
| **Text** | DistilBERT on generated action captions | 256 (projected) |
| **Audio** | gTTS → MFCC (mean/std/max) | 256 (projected) |

All three 256-d projections are concatenated → 768-d → MLP classifier.

> **Prerequisites:** Run notebooks 04 and 05 first.
> CNN+LSTM checkpoint must exist at `checkpoints/video/cnn_lstm_best.pth`.

## 1. Install & mount

In [ ]:
!pip install gtts librosa transformers opencv-python-headless -q

from google.colab import drive
drive.mount('/content/drive')

import os
# Frames must already exist from notebook 04
assert os.path.exists('/content/ucf_frames'), (
    'Frame folder not found. Run 04_video_cnn_lstm.ipynb first '
    'OR copy from Drive: !cp -r /content/drive/MyDrive/ContentRecognition/ucf_frames /content/'
)
print('Frames found!')

## 2. Imports

In [ ]:
import os, random, time, tempfile
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import DistilBertTokenizer, DistilBertModel
from gtts import gTTS
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Config

In [ ]:
CLASS_NAMES = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]
NUM_CLASSES      = len(CLASS_NAMES)
FRAMES_PER_VIDEO = 16
BATCH_SIZE       = 8
EPOCHS           = 15

FRAMES_DIR  = '/content/ucf_frames'
BASE_DIR    = '/content/drive/MyDrive/ContentRecognition'
CKPT_DIR    = f'{BASE_DIR}/checkpoints/video'
RESULTS_DIR = f'{BASE_DIR}/results/video'
CNN_LSTM_CKPT = f'{CKPT_DIR}/cnn_lstm_best.pth'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CNN_LSTM_CKPT), (
    f'CNN+LSTM checkpoint not found at {CNN_LSTM_CKPT}. '
    'Run 04_video_cnn_lstm.ipynb first!'
)
print('Config OK.')

## 4. Text generator (same as notebook 05)

In [ ]:
STOP_WORDS = [
    'the', 'a', 'an', 'this', 'that', 'is', 'are', 'was',
    'in', 'on', 'at', 'with', 'and', 'of', 'for', 'to',
    'showing', 'featuring', 'captured', 'seen', 'during'
]

TEMPLATES = [
    'this is {adj1} footage of {cls} {action}',
    'the video is showing {adj1} {cls} {action} in {place}',
    'captured footage featuring {cls} {action} with {adj2} conditions',
    'this {adj1} video is of {cls} {action} during {time}',
    'footage showing {adj1} and {adj2} {cls} {action}',
]

CLASS_SPEECH_WORDS = {
    'Basketball'  : {'adj1': ['professional','competitive','indoor','live'],
                     'adj2': ['intense','exciting','fast','dynamic'],
                     'action': ['players dribbling and shooting','match in progress','game underway'],
                     'place': ['indoor court','basketball arena','sports hall'],
                     'time' : ['the match','halftime','championship game']},
    'Biking'      : {'adj1': ['outdoor','fast','competitive','scenic'],
                     'adj2': ['challenging','rough','open','clear'],
                     'action': ['rider cycling on road','person riding bicycle','cyclist in motion'],
                     'place': ['open road','cycling trail','outdoor track'],
                     'time' : ['the race','morning ride','training session']},
    'Bowling'     : {'adj1': ['indoor','competitive','professional','casual'],
                     'adj2': ['precise','focused','clear','controlled'],
                     'action': ['player throwing ball down lane','bowler aiming at pins','strike in progress'],
                     'place': ['bowling alley','indoor lane','sports center'],
                     'time' : ['the game','tournament','practice session']},
    'CliffDiving' : {'adj1': ['extreme','breathtaking','outdoor','dangerous'],
                     'adj2': ['high','rocky','steep','dramatic'],
                     'action': ['athlete jumping from cliff','diver leaping into water','freefall in progress'],
                     'place': ['rocky cliff','ocean shore','natural diving spot'],
                     'time' : ['the jump','competition','training']},
    'GolfSwing'   : {'adj1': ['professional','outdoor','precise','calm'],
                     'adj2': ['focused','clean','open','green'],
                     'action': ['golfer swinging club','player hitting ball','golf shot in progress'],
                     'place': ['golf course','open fairway','green'],
                     'time' : ['the round','tournament','practice']},
    'HorseRiding' : {'adj1': ['outdoor','graceful','competitive','rural'],
                     'adj2': ['open','natural','calm','scenic'],
                     'action': ['rider on horseback','person galloping on horse','equestrian in motion'],
                     'place': ['open field','equestrian track','countryside'],
                     'time' : ['the race','training','competition']},
    'Skiing'      : {'adj1': ['winter','fast','outdoor','snowy'],
                     'adj2': ['steep','cold','icy','downhill'],
                     'action': ['skier going down slope','person skiing at speed','downhill skiing in progress'],
                     'place': ['snowy mountain','ski slope','winter resort'],
                     'time' : ['the run','competition','training session']},
    'Surfing'     : {'adj1': ['outdoor','exciting','coastal','extreme'],
                     'adj2': ['large','powerful','ocean','clear'],
                     'action': ['surfer riding wave','person on surfboard','wave surfing in progress'],
                     'place': ['ocean shore','coastal waters','beach'],
                     'time' : ['the surf','competition','morning session']},
    'TennisSwing' : {'adj1': ['competitive','outdoor','professional','fast'],
                     'adj2': ['precise','powerful','focused','clean'],
                     'action': ['player swinging racket','tennis shot in progress','serve and volley'],
                     'place': ['tennis court','outdoor court','sports arena'],
                     'time' : ['the match','tournament','practice session']},
    'SkateBoarding': {'adj1': ['outdoor','extreme','urban','fast'],
                      'adj2': ['technical','smooth','urban','open'],
                      'action': ['skater performing tricks','person riding skateboard','skateboard in motion'],
                      'place': ['skate park','urban area','outdoor ramp'],
                      'time' : ['the session','competition','practice']},
}


def generate_speech_text(cls_name):
    words    = CLASS_SPEECH_WORDS[cls_name]
    template = random.choice(TEMPLATES)
    sentence = template.format(
        cls    = cls_name.lower(),
        adj1   = random.choice(words['adj1']),
        adj2   = random.choice(words['adj2']),
        action = random.choice(words['action']),
        place  = random.choice(words['place']),
        time   = random.choice(words['time'])
    )
    word_list = sentence.split()
    for _ in range(random.randint(2, 4)):
        pos = random.randint(0, len(word_list))
        word_list.insert(pos, random.choice(STOP_WORDS))
    return ' '.join(word_list)


print('Text generator ready.')

## 5. Text and audio feature extractors

In [ ]:
# ── DistilBERT text encoder ───────────────────────────────────────
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
for p in bert_model.parameters():
    p.requires_grad = False
bert_model.eval()


@torch.no_grad()
def get_text_features(text):
    """Returns (1, 768) CLS embedding."""
    tokens = tokenizer(text, return_tensors='pt', padding=True,
                       truncation=True, max_length=64).to(device)
    return bert_model(**tokens).last_hidden_state[:, 0, :]


# ── gTTS → MFCC audio encoder ─────────────────────────────────────
def text_to_mfcc(text, n_mfcc=40, max_len=128):
    """Returns FloatTensor of shape (120,). Zeros on error."""
    try:
        tts = gTTS(text=text, lang='en', slow=False)
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
            tmp = f.name
        tts.save(tmp)
        audio, sr = librosa.load(tmp, sr=22050)
        os.unlink(tmp)

        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_len:
            mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :max_len]

        feat = np.concatenate([np.mean(mfcc, 1), np.std(mfcc, 1), np.max(mfcc, 1)])
        return torch.FloatTensor(feat)
    except Exception as e:
        print(f'[audio] Error: {e}')
        return torch.zeros(n_mfcc * 3)


print('Text encoder (DistilBERT) and Audio encoder (gTTS→MFCC) ready.')

## 6. Dataset — video frames + text + audio

In [ ]:
class TrimodalDataset(Dataset):
    """
    Returns (video_tensor, text_feat, audio_feat, label).

    During training : text and audio descriptions are randomly generated each call.
    During val      : fixed seed per sample index → reproducible descriptions.

    NOTE: num_workers must be 0 because DistilBERT and gTTS are not
    multiprocess-safe in PyTorch DataLoader workers.
    """
    def __init__(self, samples, transform=None, training=False):
        self.samples  = samples
        self.transform= transform
        self.training = training

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        cls_name          = CLASS_NAMES[label]

        # ── Video frames ──────────────────────────────────────────
        frame_files = sorted(os.listdir(video_path))
        frames = []
        for fname in frame_files:
            img = Image.open(os.path.join(video_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)
        video_tensor = torch.stack(frames)              # (16, 3, 224, 224)

        # ── Text features ─────────────────────────────────────────
        if not self.training:
            rng = random.getstate()
            random.seed(idx)
        text_str  = generate_speech_text(cls_name)
        text_feat = get_text_features(text_str).squeeze(0).cpu()  # (768,)

        # ── Audio features ────────────────────────────────────────
        if not self.training:
            random.seed(idx + 10000)   # Different seed for audio vs text
        audio_str  = generate_speech_text(cls_name)
        audio_feat = text_to_mfcc(audio_str).cpu()               # (120,)

        if not self.training:
            random.setstate(rng)       # Restore RNG so shuffle isn't affected

        return video_tensor, text_feat, audio_feat, label


# ── Transforms ────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Build samples list ────────────────────────────────────────────
class_to_idx = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}
all_samples  = []
for cls in CLASS_NAMES:
    cls_path = os.path.join(FRAMES_DIR, cls)
    if not os.path.exists(cls_path):
        print(f'WARNING: missing {cls}')
        continue
    for vid in os.listdir(cls_path):
        vp = os.path.join(cls_path, vid)
        if os.path.isdir(vp) and len(os.listdir(vp)) == FRAMES_PER_VIDEO:
            all_samples.append((vp, class_to_idx[cls]))

random.seed(42)
random.shuffle(all_samples)
split         = int(0.8 * len(all_samples))
train_samples = all_samples[:split]
val_samples   = all_samples[split:]

train_ds = TrimodalDataset(train_samples, train_transform, training=True)
val_ds   = TrimodalDataset(val_samples,   val_transform,   training=False)

# IMPORTANT: num_workers=0 — BERT + gTTS require main process only
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Total: {len(all_samples)} videos | Train: {len(train_ds)} | Val: {len(val_ds)}')

## 7. Trimodal Fusion Model

```
Video  → ResNet50 (layer4 unfrozen) → LSTM → 512-d → visual_proj → 256-d ─┐
Text   → DistilBERT (frozen)        → CLS        → text_proj   → 256-d ──┤→ concat → 768 → MLP → 10
Audio  → MFCC (120-d)               → MLP        → audio_proj  → 256-d ─┘
```

The visual CNN (ResNet50) backbone is **re-initialised from scratch** here and loaded
from the CNN+LSTM checkpoint — so we reuse the already-trained weights rather
than starting over.

In [ ]:
class TrimodalFusionModel(nn.Module):
    def __init__(self, num_classes=10, hidden_size=512, num_layers=2):
        super().__init__()

        # ── Visual branch: ResNet50 + LSTM ────────────────────────
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for p in resnet.parameters():
            p.requires_grad = False
        for name, p in resnet.named_parameters():
            if 'layer4' in name:
                p.requires_grad = True
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])  # output: (B*T, 2048, 1, 1)

        self.lstm = nn.LSTM(
            input_size=2048, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, dropout=0.3
        )
        self.visual_proj = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.ReLU(), nn.Dropout(0.3)
        )

        # ── Text branch: DistilBERT projection ───────────────────
        self.text_proj = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3)
        )

        # ── Audio branch: MFCC MLP ────────────────────────────────
        self.audio_proj = nn.Sequential(
            nn.Linear(120, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.3)
        )

        # ── Fusion MLP: 256 + 256 + 256 = 768 → classes ─────────
        self.fusion = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, video, text_feat, audio_feat):
        # ── Visual path ───────────────────────────────────────────
        B, T, C, H, W = video.shape
        x = video.view(B * T, C, H, W)
        with torch.no_grad():
            # CNN backbone is partially frozen; no_grad speeds things up
            cnn_out = self.cnn(x)                       # (B*T, 2048, 1, 1)
        cnn_out     = cnn_out.view(B, T, -1)            # (B, T, 2048)
        lstm_out, _ = self.lstm(cnn_out)                # (B, T, 512)
        v           = self.visual_proj(lstm_out[:, -1, :])  # (B, 256)

        # ── Text path ─────────────────────────────────────────────
        if text_feat.dim() == 3:
            text_feat = text_feat.squeeze(1)            # (B, 768)
        t = self.text_proj(text_feat)                   # (B, 256)

        # ── Audio path ────────────────────────────────────────────
        a = self.audio_proj(audio_feat)                 # (B, 256)

        # ── Concat fusion ─────────────────────────────────────────
        fused = torch.cat([v, t, a], dim=1)             # (B, 768)
        return self.fusion(fused)                       # (B, num_classes)


trimodal_model = TrimodalFusionModel(num_classes=NUM_CLASSES).to(device)
trainable = sum(p.numel() for p in trimodal_model.parameters() if p.requires_grad)
print(f'Trimodal Fusion Model ready!  Trainable params: {trainable:,}')

## 8. Load CNN+LSTM pre-trained visual weights
We transfer the CNN and LSTM weights that were already trained in notebook 04.
This means the visual branch starts from a strong checkpoint, not random weights.

In [ ]:
# Load the CNN+LSTM checkpoint
cnn_lstm_ckpt = torch.load(CNN_LSTM_CKPT, map_location=device)

# Extract only CNN and LSTM weights (ignore old classifier head)
visual_keys = {k: v for k, v in cnn_lstm_ckpt.items()
               if k.startswith('cnn.') or k.startswith('lstm.')}

missing, unexpected = trimodal_model.load_state_dict(visual_keys, strict=False)
print(f'Loaded {len(visual_keys)} visual weights from CNN+LSTM checkpoint.')
print(f'Missing keys (expected — new branches): {len(missing)}')
print(f'Unexpected keys: {len(unexpected)}')
print('Visual branch initialised from pre-trained weights!')

## 9. Train the trimodal fusion model

In [ ]:
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(
    filter(lambda p: p.requires_grad, trimodal_model.parameters()), lr=1e-4
)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
ckpt_path  = f'{CKPT_DIR}/trimodal_best.pth'

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 44)
    t0 = time.time()

    for phase in ['train', 'val']:
        trimodal_model.train() if phase == 'train' else trimodal_model.eval()
        loader = train_loader if phase == 'train' else val_loader

        running_loss, running_correct = 0.0, 0

        for videos, text_feats, audio_feats, labels in loader:
            videos      = videos.to(device)
            text_feats  = text_feats.to(device)
            audio_feats = audio_feats.to(device)
            labels      = labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs  = trimodal_model(videos, text_feats, audio_feats)
                loss     = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss    += loss.item() * videos.size(0)
            running_correct += torch.sum(preds == labels)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = running_correct.double() / len(loader.dataset)

        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc.item())
        print(f'  {phase.upper():5} → Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_val_acc:
            best_val_acc = epoch_acc
            torch.save(trimodal_model.state_dict(), ckpt_path)
            print(f'  ✅ Best saved! Val Acc: {best_val_acc:.4f}')

    print(f'  ⏱  {time.time()-t0:.1f}s')

print(f'\nTraining complete! Best Val Acc: {best_val_acc:.4f}')

## 10. Evaluate and plot

In [ ]:
trimodal_model.load_state_dict(torch.load(ckpt_path, map_location=device))
trimodal_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for videos, text_feats, audio_feats, labels in val_loader:
        outputs  = trimodal_model(
            videos.to(device),
            text_feats.to(device),
            audio_feats.to(device)
        )
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 5))

axes[0].plot(history['train_acc'],  label='Train', marker='o', color='#8E44AD')
axes[0].plot(history['val_acc'],    label='Val',   marker='o', color='#E67E22')
axes[0].set_title('Trimodal Fusion — Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train', marker='o', color='#8E44AD')
axes[1].plot(history['val_loss'],   label='Val',   marker='o', color='#E67E22')
axes[1].set_title('Trimodal Fusion — Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[2])
axes[2].set_title('Confusion Matrix — Trimodal', fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
plt.setp(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/trimodal_results.png', dpi=150)
plt.show()
print('Results saved!')

## 11. Final comparison — CNN+LSTM vs Trimodal Fusion

In [ ]:
# Update cnn_lstm_val_acc with the value printed at the end of notebook 04
cnn_lstm_val_acc  = None   # e.g. 0.9737 — paste your result here
trimodal_val_acc  = best_val_acc

print(f'\n{"="*48}')
print(f'{"MODEL":<28} {"VAL ACCURACY":>15}')
print(f'{"="*48}')
if cnn_lstm_val_acc:
    print(f'{"CNN+LSTM (video only)":<28} {cnn_lstm_val_acc*100:>14.2f}%')
print(f'{"Trimodal (video+text+audio)":<28} {trimodal_val_acc*100:>14.2f}%')
print(f'{"="*48}')
if cnn_lstm_val_acc:
    diff = (trimodal_val_acc - cnn_lstm_val_acc) * 100
    sign = '+' if diff >= 0 else ''
    print(f'Fusion improvement: {sign}{diff:.2f}%')